In [2]:
# Étape 1 : Installation des bibliothèques nécessaires
!pip install sentence-transformers tqdm

# Importations après l'installation pour s'assurer qu'elles sont disponibles
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
import json
from google.colab import files
from datetime import datetime

# Étape 3 : Définition de la fonction de découpage du texte en morceaux (chunks)
def chunk_text(text, chunk_size=500, chunk_overlap=100):
    chunks = []
    text = ' '.join(text.split())  # Nettoyage des espaces/sauts de ligne multiples

    start_index = 0
    while start_index < len(text):
        end_index = start_index + chunk_size
        chunk = text[start_index:end_index]
        chunks.append({
            "chunk_text": chunk.strip()
        })
        start_index += (chunk_size - chunk_overlap)
        if end_index >= len(text):
            break

    # Suppression des chunks vides ou dupliqués
    final_chunks = []
    seen = set()
    for c in chunks:
        if c["chunk_text"] and c["chunk_text"] not in seen:
            final_chunks.append(c)
            seen.add(c["chunk_text"])

    return final_chunks

# Définition des paramètres de chunking
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

# Étape 4 : Chargement du modèle d'embeddings
print("Chargement du modèle d'embeddings. Cela peut prendre un instant...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Modèle chargé avec succès.")


# Fonction pour traiter une question complète (Étapes 2 à 5)
def process_question():
    # Étape 2 : Saisie de votre question
    question_text = input("Tapez votre question ou votre texte : ")
    print(f"\nTexte saisi ({len(question_text)} caractères) :")
    print(question_text)

    # Découpage du texte en morceaux (chunks)
    text_chunks = chunk_text(question_text, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    print(f"Nombre de chunks créés : {len(text_chunks)}")
    for i, chunk in enumerate(text_chunks):
        print(f"--- Chunk {i+1} ---")
        print(chunk["chunk_text"])

    # Génération des embeddings pour chaque chunk
    for chunk_data in tqdm(text_chunks, desc="Génération des embeddings"):
        embedding = model.encode(chunk_data["chunk_text"]).tolist()
        chunk_data["embedding"] = embedding

    print(f"Embeddings générés pour {len(text_chunks)} chunks.")
    if text_chunks:
        print("\nAperçu du premier chunk avec son embedding :")
        print(f"Texte du chunk : {text_chunks[0]['chunk_text']}")
        print(f"Embedding (premières 5 valeurs) : {text_chunks[0]['embedding'][:5]}...")

    # Étape 5 : Création et téléchargement du fichier JSON
    json_output = []
    for chunk_data in text_chunks:
        json_output.append({
            "chunk_text": chunk_data["chunk_text"],
            "embedding": chunk_data["embedding"]
        })

    output_filename = f"question_chunks_with_embeddings_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(json_output, f, ensure_ascii=False, indent=4)

    print(f"Fichier JSON '{output_filename}' créé avec succès.")
    files.download(output_filename)
    print("Processus terminé ! Le fichier JSON devrait être téléchargé sur votre ordinateur.")


# Exécute le processus pour la première question
process_question()

# (Optionnel) Étape 6 : Si vous voulez traiter plusieurs questions à la suite,
# vous pouvez appeler process_question() de nouveau ici ou dans une nouvelle cellule.
# Par exemple, pour demander une autre question après la première :
# process_question()

Chargement du modèle d'embeddings. Cela peut prendre un instant...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modèle chargé avec succès.
Tapez votre question ou votre texte : Est-ce que je peux construire une piscine sur ma parcelle ?

Texte saisi (59 caractères) :
Est-ce que je peux construire une piscine sur ma parcelle ?
Nombre de chunks créés : 1
--- Chunk 1 ---
Est-ce que je peux construire une piscine sur ma parcelle ?


Génération des embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings générés pour 1 chunks.

Aperçu du premier chunk avec son embedding :
Texte du chunk : Est-ce que je peux construire une piscine sur ma parcelle ?
Embedding (premières 5 valeurs) : [-0.031101424247026443, 0.03485848382115364, 0.008615672588348389, -0.041339945048093796, -0.018697354942560196]...
Fichier JSON 'question_chunks_with_embeddings_20260730_144538.json' créé avec succès.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Processus terminé ! Le fichier JSON devrait être téléchargé sur votre ordinateur.
